### Section wise Chunking of PDFs for LLM Applications

In [ ]:
import fitz
from bs4 import BeautifulSoup 
import csv
import re
import json
import os

def ConvertPdfToHTML(filepath, filename):
    total_pages = 0
    doc = fitz.open(filepath)
    text = ''
    skip = []
    for i, page in enumerate(doc):
        if i in skip:
            continue
        else:
            text += "<h5>Page Number: "+str(i+1)+"</h5>"+"\n"+ page.get_text("html")
            total_pages = i+1
    with open(os.getcwd()+f"/HTML/{filename}.html", "w") as fp:
        fp.write(text)
    doc.close()
    return text,total_pages


def CreateSectionWiseChunks(filename, extractedText, divTag, headerTag, headcount, footercount, tagToRemove, total_pages, separator='|'):
    soup = BeautifulSoup(extractedText, 'html5lib')
    result = []
    heading = ""
    allHeadingList = []
    allChunkList = []
    allPageNumberList = []
    f = -1
    heading_regex = r'<b>(.*?)<\/b>'
    chunk = ""
    if divTag:
        divs = soup.find_all('div')
        for top_idx,j in enumerate(divs):
            page_number = "".join(str(soup.select('h5')[top_idx].text).splitlines())
            newhead=headcount
            entries = len(j.find_all("p"))
            for idx,i in enumerate(j.find_all('p')):
                if entries-footercount-1 == i:
                    break
                if newhead!=0:
                    newhead-=1
                    continue
                if i.find_all('b'): # Change this condition for Detecting Heading (Heading content)
                    if f == 0:
                        res = "".join([x.text.strip() for x in i.find_all('span') if x.parent.name == "b"])
                        res = re.sub(r'[^\x00-\x7F]+',' ', res)
                        if res.strip():
                            heading += " "+ res
                    else:
                        if heading.strip():
                            result.append([filename+separator+heading+separator+chunk,page_number])#(Combines Heading, Context into single paragraph)
                            allHeadingList.append(heading)
                            allPageNumberList.append(page_number)
                        res = "".join([x.text.strip() for x in i.find_all('span') if x.parent.name == "b"])
                        res = re.sub(r'[^\x00-\x7F]+',' ', res)
                        heading = res
                        chunk = ""
                        f = 0
                    x = "".join([x.text.strip() for x in i.find_all("span") if x.parent.name != "b" and not x.find("b")])
                    if x.strip():
                        f = 1
                        if x != "":
                            x = re.sub(r'[^\x00-\x7F]+',' ', x)
                            chunk += x
                            allChunkList.append(chunk)
                else: # Checks and splits into contexts (Pragaraph content)
                    f = 1
                    x = "".join([x.text.strip() for x in i.find_all("span") if x.parent.name != "b" and not x.find("b")])
                    x = x.strip()
                    if x != "":
                        x = re.sub(r'[^\x00-\x7F]+',' ', x)
                        chunk += x
                        allChunkList.append(chunk)
        if len(heading) !=0:
            if heading.strip():
                result.append([filename+separator+heading+separator+chunk,page_number])#(Combines Heading, Context into single paragraph)
                allHeadingList.append(heading)
                allPageNumberList.append(page_number)
            res = "".join([x.text.strip() for x in i.find_all('span') if x.parent.name == "b"])
            print("res: ", res)
            res = re.sub(r'[^\x00-\x7F]+',' ', res)
            heading = res
            chunk = ""
            f = 0

    else:
        # Creating chunks without searching for div tags (doesn't remove header and footer)
        tags = soup.find_all('p')
        f = 0
        for i in tags:
            if i.find(f'{headerTag}'):
                if f == 0:
                    heading += "".join(str(i.find('span').text).splitlines())
                else:
                    result.append([chunk])
                    heading = "".join(str(i.find('span').text).splitlines())
                    chunk = ""
                f = 0
            else:
                f = 1
                x = " ".join(str(i.find('span').text).splitlines())
                x = x.strip()
                if x != "":
                    x = re.sub(r'[^\x00-\x7F]+',' ', x)
                    chunk += x
    print(allChunkList)
    FinalResult = {
        "all_headings": allHeadingList,
        "all_chunks": allChunkList,
        "heading_section_chunks": result,
        "document_reference": filename,
        "total_pages": total_pages,
        "total_headings": len(allHeadingList)
    }
    return FinalResult


def GenerateChunks_csv(result,filename):
    with open(os.getcwd()+f"/CSV/{filename}.csv", 'w', encoding="utf-8") as file:
        csvwriter = csv.writer(file)
        csvwriter.writerows(result)



In [32]:
###########  TABLE EXTRACTION METHOD 2 ##############
import fitz
import pdfplumber
from collections import defaultdict

def getPagesWithTable(filepath):
    with pdfplumber.open(filepath) as f:
        page_with_tables = []
        for i,j in enumerate(f.pages):
            # print(i.extract_tables())
            if(j.find_table()):
                page_with_tables.append(i+1)
    return page_with_tables

def extract_table(range_start, range_end,filepath, has_headers = False):
    count = 0
    with pdfplumber.open(filepath) as pdf:
        all_tables = defaultdict(dict)
        for page_nums in range(range_start-1, range_end):
            page = pdf.pages[page_nums]
            tables = page.extract_tables()
            # layout = page.layout
            # print(layout)
            for tbx, table in enumerate(tables,start=1):
                count +=1
                table_data = []
                if has_headers:
                    cols = [str(col) for col in table[0]]
                    table_data = [{cols[i]: row[i] for i in range(len(cols))} for row in table[1:]]
                else:
                    cols = [f"col{i+1}" for i in range(len(table[0]))]
                    table_data = [{cols[i]: row[i] for i in range(len(cols))} for row in table]
                table_key = f"Table_{count}"
                table_info = {"Table_Index": tbx, "Page_Number": page_nums+1}
                all_tables[table_key]["TableData"] = table_data
                all_tables[table_key]["TableInfo"] = table_info
    return dict(all_tables)

def extract_table_list(range_start, range_end,filepath, has_headers = False):
    count = 0
    with pdfplumber.open(filepath) as pdf:
        all_tables = defaultdict(dict)
        for page_nums in range(range_start-1, range_end):
            page = pdf.pages[page_nums]
            tables = page.extract_tables()
            for tbx, table in enumerate(tables,start=1):
                count +=1
                table_data = []
                if has_headers:
                    cols = [str(col) for col in table[0]]
                    table_data = [{cols[i]: row[i] for i in range(len(cols))} for row in table[1:]]
                else:
                    cols = [f"col{i+1}" for i in range(len(table[0]))]
                    table_data = [row[i] for i in range(len(cols)) for row in table]
                table_key = f"Table_{count}"
                table_info = {"Table_Index": tbx, "Page_Number": page_nums+1}
                all_tables[table_key]["TableData"] = table_data
                all_tables[table_key]["TableInfo"] = table_info
    return dict(all_tables)

In [33]:
import json
import os
if __name__ == "__main__":
    filename = "currency_internationalisation"
    filepath = os.getcwd()+"/PDF/currency_internationisation.pdf"
    # Step 1: Convert PDF to HTML
    extracted_text, totalpages = ConvertPdfToHTML(filepath, filename)
    print(extracted_text, totalpages)
    # Step 2: Extract Text from HTML
    chunkedData = CreateSectionWiseChunks(filename, extracted_text, 'div', 'b', 0, 0,'p',totalpages,'|')
    print(chunkedData['heading_section_chunks'])
    dataToLoad = chunkedData['heading_section_chunks']
    # Step 3: Write section wise chunks into csv
    GenerateChunks_csv(dataToLoad, filename)
    print(chunkedData)
    # Step 4: Check If tables exist in pdf
    pages = getPagesWithTable(filepath)
    print(pages)
    istable = False
    if (istable):
        page_table = extract_table_list(0,79,filepath)
        print(page_table)
        # print(page_table)


<h5>Page Number: 1</h5>
<div id="page0" style="width:595.0pt;height:842.0pt">
<p style="top:796.5pt;left:85.4pt;line-height:7.0pt"><span style="font-family:Arial,sans-serif;font-size:7.0pt;color:#000000">BIS Papers No 61  </span></p>
<p style="top:794.5pt;left:534.1pt;line-height:9.0pt"><span style="font-family:Arial,sans-serif;font-size:9.0pt;color:#000000">9</span></p>
<p style="top:803.8pt;left:85.1pt;line-height:1.0pt"><span style="font-family:Arial,sans-serif;font-size:1.0pt;color:#ffffff"> </span></p>
<p style="top:804.9pt;left:85.1pt;line-height:1.0pt"><span style="font-family:Arial,sans-serif;font-size:1.0pt;color:#ffffff"> </span></p>
<p style="top:72.9pt;left:160.1pt;line-height:15.0pt"><b><span style="font-family:Arial,sans-serif;font-size:15.0pt;color:#000000">Currency internationalisation: an overview </span></b></p>
<p style="top:119.6pt;left:274.1pt;line-height:11.0pt"><span style="font-family:Arial,sans-serif;font-size:11.0pt;color:#000000">Peter B Kenen</span><sup><spa